[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/huggingface-nlp-certified/notebooks/day-13-review-advanced-patterns.ipynb#scrollTo=a1b2c3d4)

---
# Day 13 · Review — Advanced Patterns, Common Pitfalls, and Production Readiness
**certified-journeys / huggingface-nlp-certified** · Day 13 · Review

> **Goal for today:** Audit your Days 1–12 work, identify the most common transformer pitfalls, and produce a production-readiness checklist you can apply to any future fine-tuning project.

## Step 1 · Transformers Best Practices — Pipeline API Overview

The `pipeline()` API from 🤗 Transformers is the recommended entry point for inference. It handles:
- Tokenization (including padding and truncation)
- Model forward pass
- Post-processing (softmax, argmax, decoding)

Read the official guide: [https://huggingface.co/docs/transformers/pipeline_tutorial](https://huggingface.co/docs/transformers/pipeline_tutorial)

| Concern | Rule of thumb |
|---|---|
| Task selection | Use `task=` string; don't infer from model name |
| Batching | Set `batch_size` explicitly; default=1 is slow |
| Device | Pass `device=0` for GPU; `device="cpu"` if no GPU |
| Max length | Always pass `truncation=True, max_length=512` |
| Model card | Check the model card before using any Hub checkpoint |

In [ ]:
%pip install -q transformers datasets evaluate accelerate torch

In [ ]:
from transformers import pipeline

# Best-practice usage of pipeline with explicit settings
classifier = pipeline(
    task="text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device="cpu",          # Use device=0 in Colab with GPU runtime
    truncation=True,
    max_length=512,
)

samples = [
    "The tokenizer saved alongside the model is absolutely critical.",
    "Forgetting to save the tokenizer causes silent mismatches at inference.",
    "Fine-tuning went smoothly — loss converged within 3 epochs.",
]

# Batch inference is far faster than calling the pipeline one item at a time
results = classifier(samples, batch_size=2)
for text, result in zip(samples, results):
    print(f"[{result['label']} | {result['score']:.3f}] {text[:60]}...")

### What just happened?
- The pipeline loaded the tokenizer and model from the Hub in one call.
- **`batch_size=2` processed two inputs in one forward pass** — typically 2–4× faster than serial inference.
- `truncation=True` prevents "Token indices sequence length longer than the specified maximum" errors that often appear silently in production.
- The output `label` and `score` come from the model's config — no manual decoding needed.

## Step 2 · Tokenizer Audit — `pad_token` vs `eos_token`

BERT-family models have a dedicated `[PAD]` token. GPT-2 and other causal LMs do **not** — they were not trained with padding. This creates a subtle bug:

```python
# WRONG for GPT-2
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer(batch, padding=True)  # raises ValueError: pad_token is not set

# CORRECT — use eos_token as pad_token
tokenizer.pad_token = tokenizer.eos_token
```

| Model family | Has `pad_token`? | Fix needed? |
|---|---|---|
| BERT, DistilBERT, RoBERTa | Yes (`[PAD]`) | No |
| GPT-2, GPT-Neo, Falcon | No | Yes: `pad_token = eos_token` |
| LLaMA 2 / Mistral | No | Yes: `pad_token = eos_token` |
| T5 | Yes (`<pad>`) | No |

> **Why is this a silent bug?** If you set `pad_token_id` in the model config but do not also fix the tokenizer, the tokenizer will raise an error or use `unk_token` as a fallback — producing garbled attention masks.

In [ ]:
from transformers import AutoTokenizer

# ── BERT: pad_token already exists ──────────────────────────────────────────
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
print(f"BERT pad_token : {bert_tok.pad_token!r}  (id={bert_tok.pad_token_id})")
print(f"BERT eos_token : {bert_tok.eos_token!r}")

# ── GPT-2: no pad_token — must set it before batch inference ────────────────
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
print(f"\nGPT-2 pad_token (before fix) : {gpt2_tok.pad_token!r}")

# Apply the required fix
gpt2_tok.pad_token = gpt2_tok.eos_token
print(f"GPT-2 pad_token (after fix)  : {gpt2_tok.pad_token!r}  (id={gpt2_tok.pad_token_id})")

# Verify batched tokenization now works
batch = ["Hello world", "A much longer sentence that would require padding"]
encoded = gpt2_tok(batch, padding=True, return_tensors="pt")
print(f"\nBatch input_ids shape : {encoded['input_ids'].shape}")
print(f"Attention mask shape  : {encoded['attention_mask'].shape}")

### What just happened?
- BERT has `[PAD]` (token id 0) built in — no fix required.
- **GPT-2's `pad_token` is `None` by default** — batched tokenization raises `ValueError` without the fix.
- After setting `pad_token = eos_token`, the attention mask correctly marks padding positions as 0, so the model ignores them.
- The model config also needs updating: `model.config.pad_token_id = tokenizer.eos_token_id` before any generation call.

## Step 3 · Debugging the Label Shift in Causal LM Training

Causal language models (GPT-2, LLaMA) are trained to **predict the next token**. During training:
- `input_ids`  = `[t0, t1, t2, ..., tN-1]`
- `labels`     = `[t1, t2, t3, ..., tN]`  ← shifted left by one

If you forget the shift, the model tries to predict the token it just saw — trivial memorisation, not language modelling.

```
input_ids : The  cat  sat  on  the  mat
labels    :  cat  sat  on  the  mat  <EOS>
              ↑
              loss computed here (position 0 predicts position 1)
```

The `DataCollatorForLanguageModeling` handles this automatically. The shape-mismatch error appears when you manually assign `labels = input_ids` **before** the collator runs — the collator then shifts again, producing wrong cross-entropy.

| Mistake | Symptom |
|---|---|
| Manual shift + collator shift | Loss NaN or stuck |
| No shift at all | Loss near zero from epoch 1 (overfitting to current token) |
| Wrong `ignore_index` on padding | Loss includes pad positions — artificially low |
| Shape `[B, T]` vs `[B, T, V]` | `CrossEntropyLoss` dimension error |

In [ ]:
import torch
import torch.nn.functional as F

# Simulate a batch of 2 sequences, length 6, vocab size 50 (toy)
BATCH, SEQ_LEN, VOCAB = 2, 6, 50
torch.manual_seed(42)

logits = torch.randn(BATCH, SEQ_LEN, VOCAB)  # model output shape
input_ids = torch.randint(0, VOCAB, (BATCH, SEQ_LEN))

# ── WRONG: labels == input_ids (no shift) ───────────────────────────────────
labels_wrong = input_ids.clone()
# CrossEntropyLoss expects logits shape [B*T, V] and labels shape [B*T]
loss_wrong = F.cross_entropy(
    logits.view(-1, VOCAB),
    labels_wrong.view(-1)
)
print(f"Loss (wrong — no shift)  : {loss_wrong.item():.4f}")

# ── CORRECT: shift labels left by one, ignore the last position ─────────────
# input  : [t0, t1, t2, t3, t4, t5]
# labels : [t1, t2, t3, t4, t5, -100]  (-100 is ignored by CrossEntropyLoss)
labels_correct = input_ids.clone()
labels_correct = torch.cat(
    [labels_correct[:, 1:], torch.full((BATCH, 1), -100)],
    dim=1
)
loss_correct = F.cross_entropy(
    logits.view(-1, VOCAB),
    labels_correct.view(-1),
    ignore_index=-100   # padding positions do not contribute to the loss
)
print(f"Loss (correct — shifted) : {loss_correct.item():.4f}")

print("\nLabel shift demo:")
print(f"  input_ids row 0  : {input_ids[0].tolist()}")
print(f"  labels_correct 0 : {labels_correct[0].tolist()}  (-100 = ignored)")

### What just happened?
- Without the shift the model is asked to predict `t0` from `t0` — it can memorise trivially, giving a deceptively low loss.
- **The correct label for position `i` is `input_ids[i+1]`**, not `input_ids[i]`.
- `ignore_index=-100` tells `CrossEntropyLoss` to skip padding and the sentinel last position — without this, those positions contribute nonsense gradient signal.
- In practice, `DataCollatorForLanguageModeling(tokenizer, mlm=False)` does the shift automatically — only write it manually when debugging.

## Step 4 · The Three Most Common OOM Errors in Transformer Fine-Tuning

Out-of-memory (OOM) crashes are the #1 frustration for new fine-tuners. Here are the three root causes and their fixes:

### OOM 1 — Batch size too large
Each forward pass stores activations for every sample. On a T4 GPU (16 GB), a `bert-base` fine-tune typically maxes out at `batch_size=32` for 128-token sequences.

**Fix:** Halve `per_device_train_batch_size` and double `gradient_accumulation_steps` to keep the effective batch size the same.

### OOM 2 — Sequence length too long
Memory scales **quadratically** with sequence length in standard attention. A sequence of 1024 tokens needs ~4× the GPU memory of a 512-token sequence.

**Fix:** Set `max_length=256` or `max_length=128` during tokenization, or use a model with FlashAttention.

### OOM 3 — No gradient accumulation
Without `gradient_accumulation_steps`, every micro-batch update triggers an optimizer step, keeping all optimizer states in memory simultaneously.

**Fix:** Set `gradient_accumulation_steps=4` or higher to simulate a larger batch while staying memory-efficient.

| Problem | Memory impact | Fix |
|---|---|---|
| Batch size 64 | 2× vs batch 32 | Halve it; accumulate |
| Sequence 1024 | 4× vs seq 512 | Truncate to 256–512 |
| `fp32` training | 2× vs `fp16` | Set `fp16=True` in `TrainingArguments` |
| Optimizer states | Steady baseline | Use `optim="adamw_8bit"` (bitsandbytes) |

In [ ]:
from transformers import TrainingArguments

# ── Memory-efficient TrainingArguments template ──────────────────────────────
# These settings let you train bert-base on a 16 GB GPU with 512-token sequences
args = TrainingArguments(
    output_dir="./output/memory-efficient-run",
    per_device_train_batch_size=8,      # small micro-batch to stay within VRAM
    gradient_accumulation_steps=4,      # effective batch = 8 * 4 = 32
    fp16=True,                          # halves activation memory on Nvidia GPUs
    num_train_epochs=3,
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,        # requires evaluation_strategy == save_strategy
    metric_for_best_model="eval_loss",
    report_to="none",                   # disable wandb/mlflow for this demo
)

# Show the effective batch size calculation
effective_batch = (
    args.per_device_train_batch_size
    * args.gradient_accumulation_steps
)
print("TrainingArguments memory summary")
print(f"  per_device_train_batch_size : {args.per_device_train_batch_size}")
print(f"  gradient_accumulation_steps : {args.gradient_accumulation_steps}")
print(f"  effective batch size        : {effective_batch}")
print(f"  fp16                        : {args.fp16}")
print(f"  load_best_model_at_end      : {args.load_best_model_at_end}")

### What just happened?
- `gradient_accumulation_steps=4` gives you an **effective batch of 32 while storing only 8 samples** of activations in GPU memory at a time.
- `fp16=True` halves the memory footprint of weights, activations, and gradients on supported GPUs.
- **`load_best_model_at_end=True` requires `save_strategy == evaluation_strategy`** — a common config mistake that causes a confusing error at the end of training.
- `report_to="none"` prevents accidental credential prompts in automated runs.

## Step 5 · Days 1–12 Audit — Verify Every Fine-Tuning Run

Before the capstone, audit each previous notebook to confirm:

1. **`output_dir` is descriptive** — e.g. `bert-sst2-lr2e-5-ep3`, not `./output`
2. **`model.save_pretrained(output_dir)` was called** — the model weights are saved
3. **`tokenizer.save_pretrained(output_dir)` was called** alongside the model
4. **Evaluation metrics were printed** — accuracy, F1, or loss on the eval set
5. **A final `trainer.evaluate()` was run** after training completed

The cell below simulates what a saved run directory should look like and verifies it programmatically.

In [ ]:
import os
import json
from pathlib import Path

def audit_run_directory(run_dir: str) -> dict:
    """Check that a fine-tuning output directory contains all required artifacts."""
    p = Path(run_dir)
    checks = {
        "config.json":        (p / "config.json").exists(),
        "tokenizer.json":     (p / "tokenizer.json").exists(),
        "tokenizer_config.json": (p / "tokenizer_config.json").exists(),
        "model weights":      (
            (p / "pytorch_model.bin").exists()
            or (p / "model.safetensors").exists()
        ),
        "trainer_state.json": (p / "trainer_state.json").exists(),
    }
    return checks

# ── Create a mock run directory to demonstrate the audit ────────────────────
mock_dir = "/tmp/bert-sentiment-lr2e-5-ep3"
os.makedirs(mock_dir, exist_ok=True)

# Simulate the files a real Trainer would save
for fname, content in [
    ("config.json",           json.dumps({"model_type": "bert"})),
    ("tokenizer.json",        json.dumps({"version": "1.0"})),
    ("tokenizer_config.json", json.dumps({"model_max_length": 512})),
    ("model.safetensors",     ""),   # empty placeholder
    ("trainer_state.json",    json.dumps({"best_metric": 0.923})),
]:
    Path(mock_dir, fname).write_text(content)

# Run the audit
report = audit_run_directory(mock_dir)
print(f"Audit: {mock_dir}")
all_ok = True
for check, passed in report.items():
    icon = "✓" if passed else "✗"
    if not passed:
        all_ok = False
    print(f"  {icon}  {check}")
print(f"\nResult: {'PASS — all artifacts present' if all_ok else 'FAIL — missing artifacts'}")

### What just happened?
- `config.json` is written by `model.save_pretrained()` — if it is missing, the model was never saved.
- `tokenizer.json` + `tokenizer_config.json` are written by `tokenizer.save_pretrained()` — **these are the silent bug**: a model loads fine without them, but tokenization differs between the training tokenizer and whatever the user loads separately.
- **`model.safetensors` is preferred over `pytorch_model.bin`** in recent transformers — the audit checks for both.
- `trainer_state.json` records the best eval metric from `load_best_model_at_end` — always verify the metric before pushing to the Hub.

## Step 6 · Production Checklist

Before any model goes to the Hub or to a production endpoint, verify all six items below.

| # | Item | How to verify |
|---|---|---|
| 1 | **Tokenizer saved** | `tokenizer_config.json` + `tokenizer.json` in `output_dir` |
| 2 | **Model saved** | `config.json` + `model.safetensors` in `output_dir` |
| 3 | **Pushed to Hub** | `trainer.push_to_hub()` or `model.push_to_hub(repo_id)` |
| 4 | **Eval results recorded** | `trainer_state.json` or explicit `trainer.evaluate()` call |
| 5 | **Model card written** | `README.md` in the Hub repo with task, dataset, metric, and usage |
| 6 | **Inference test passed** | `pipeline(task, model=repo_id)(sample_input)` returns expected output |

> **Tip:** The most common production incident with fine-tuned models is item 6 catching a mismatch that items 1–5 missed. Always run an inference test on the **saved or Hub copy**, not on the in-memory `trainer.model`.

In [ ]:
def production_checklist(output_dir: str, hub_repo_id: str = None) -> None:
    """Print a production-readiness checklist for a fine-tuned model."""
    from pathlib import Path

    p = Path(output_dir)
    items = [
        (
            "Tokenizer saved",
            (p / "tokenizer.json").exists() and (p / "tokenizer_config.json").exists(),
            "tokenizer.save_pretrained(output_dir)",
        ),
        (
            "Model saved",
            (p / "config.json").exists()
            and ((p / "model.safetensors").exists() or (p / "pytorch_model.bin").exists()),
            "model.save_pretrained(output_dir)",
        ),
        (
            "Pushed to Hub",
            hub_repo_id is not None,
            "trainer.push_to_hub() or model.push_to_hub(repo_id)",
        ),
        (
            "Eval results recorded",
            (p / "trainer_state.json").exists(),
            "trainer.evaluate() + trainer_state.json present",
        ),
        (
            "Model card written",
            hub_repo_id is not None,  # Hub push creates a README automatically
            "README.md on Hub with task, dataset, metrics, usage snippet",
        ),
        (
            "Inference test passed",
            False,  # must be verified manually
            "pipeline(task, model=repo_id)(sample_input)",
        ),
    ]

    print(f"Production checklist for: {output_dir}")
    if hub_repo_id:
        print(f"Hub repo: {hub_repo_id}")
    print("-" * 60)
    for name, passed, fix in items:
        icon = "✓" if passed else "○"
        status = "OK" if passed else f"TODO → {fix}"
        print(f"  {icon}  {name:<30}  {status}")

# Demo with our mock directory from Step 5
production_checklist(
    output_dir="/tmp/bert-sentiment-lr2e-5-ep3",
    hub_repo_id=None,  # set to "your-username/bert-sentiment" after push
)

### What just happened?
- The checklist function is a **lightweight audit you can paste into any notebook** at the end of a fine-tuning run.
- Item 6 (Inference test) is always `False` by design — it **must be run manually** because an automated check would pass trivially on the in-memory model.
- **`hub_repo_id=None` flags both Hub push and model card as TODO** — these are the two items most commonly skipped during quick experiments.
- Run this function before calling it done; if any item is `○`, fix it before the capstone.

## Step 7 · Advanced Patterns — Saving and Reloading a Fine-Tuned Model

The correct save/load cycle is:

```python
# Save
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)   # must match the model

# Reload
from transformers import AutoModelForSequenceClassification, AutoTokenizer
model     = AutoModelForSequenceClassification.from_pretrained(output_dir)
tokenizer = AutoTokenizer.from_pretrained(output_dir)  # from same dir!
```

Common mistake: reloading the tokenizer from the **original checkpoint** instead of `output_dir`. If you added special tokens during fine-tuning (common in dialogue or instruction tuning), the original tokenizer will produce the wrong token IDs.

| Anti-pattern | Effect |
|---|---|
| `AutoTokenizer.from_pretrained("bert-base-uncased")` after fine-tune | Mismatched vocab if you added tokens |
| Loading model from Hub but tokenizer from disk | `pad_token_id` mismatch |
| Saving every checkpoint but not the final model | Deploy wrong checkpoint |
| `push_to_hub()` without a model card | Hub repo has no usage snippet |

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)
import os

# ── Simulate fine-tuning by saving a pretrained model as if it were fine-tuned
CHECKPOINT = "distilbert-base-uncased-finetuned-sst-2-english"
SAVE_DIR   = "/tmp/my-finetuned-model"

# Load and immediately save (simulating end-of-training save)
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model     = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT)

tokenizer.save_pretrained(SAVE_DIR)   # always alongside the model
model.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")
print("Files:", sorted(os.listdir(SAVE_DIR)))

# ── Correct reload — both from the same directory ────────────────────────────
tok_reload   = AutoTokenizer.from_pretrained(SAVE_DIR)
model_reload = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)

# Quick sanity-check inference
inputs = tok_reload(
    "Saving the tokenizer alongside the model is non-negotiable.",
    return_tensors="pt",
    truncation=True,
    max_length=512,
)
import torch
with torch.no_grad():
    logits = model_reload(**inputs).logits

pred_id    = logits.argmax().item()
pred_label = model_reload.config.id2label[pred_id]
print(f"\nInference test → predicted label: {pred_label}")
print("Reload cycle ✓ — tokenizer and model are consistent.")

### What just happened?
- `tokenizer.save_pretrained(SAVE_DIR)` writes `tokenizer.json`, `tokenizer_config.json`, `special_tokens_map.json`, and `vocab.txt` — together these fully reproduce the tokenizer state.
- **Both `from_pretrained` calls point to `SAVE_DIR`** — this guarantees consistency even if you added special tokens.
- The inference test confirms the full round-trip: save → reload → predict.
- In production, replace `SAVE_DIR` with the Hub repo ID after `push_to_hub()` to test the live version.

## Step 8 · Transformers Pipeline Best Practices Reference

From the [official pipeline tutorial](https://huggingface.co/docs/transformers/pipeline_tutorial), the key patterns for production-quality inference:

```python
# Pattern 1 — Explicit device
pipe = pipeline(task, model=model_id, device=0)   # GPU 0
pipe = pipeline(task, model=model_id, device="cpu") # CPU

# Pattern 2 — Chunking for long documents
pipe = pipeline(
    "text-classification",
    model=model_id,
    truncation=True,
    max_length=512,
)

# Pattern 3 — Dataset streaming (no OOM on large sets)
from datasets import load_dataset
dataset = load_dataset("csv", data_files="data.csv", split="train")
for out in pipe(KeyDataset(dataset, "text"), batch_size=32):
    process(out)

# Pattern 4 — Return all scores (not just top-1)
results = pipe(texts, return_all_scores=True)  # deprecated in v4.39+
results = pipe(texts, top_k=None)              # current API
```

| Anti-pattern | Better approach |
|---|---|
| Loop over items one by one | Pass a list; set `batch_size` |
| `return_all_scores=True` | `top_k=None` (pipeline v4.39+) |
| No explicit device | Always pass `device=` |
| No truncation | Always pass `truncation=True, max_length=N` |

In [ ]:
from transformers import pipeline
import time

# Demonstrate batch vs. serial throughput difference
texts = [
    "Transformers are the backbone of modern NLP.",
    "OOM errors are the nemesis of fine-tuning engineers.",
    "Gradient accumulation is a memory-efficient training trick.",
    "Always save the tokenizer with the model.",
    "The pipeline API abstracts away tokenization and post-processing.",
    "Fine-tuning BERT on SST-2 takes about 20 minutes on a T4.",
] * 3   # 18 texts total

pipe = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device="cpu",
    truncation=True,
    max_length=128,
)

# Serial: process one at a time
t0 = time.perf_counter()
serial_out = [pipe(t) for t in texts]
serial_time = time.perf_counter() - t0

# Batch: process all at once
t0 = time.perf_counter()
batch_out = pipe(texts, batch_size=6)
batch_time = time.perf_counter() - t0

print(f"Serial time : {serial_time:.2f}s")
print(f"Batch time  : {batch_time:.2f}s")
print(f"Speedup     : {serial_time / batch_time:.1f}×")

# top_k=None for all class scores
detailed = pipe(texts[:2], top_k=None)
print("\nAll scores for first sample:")
for score in sorted(detailed[0], key=lambda x: x['score'], reverse=True):
    print(f"  {score['label']}: {score['score']:.4f}")

### What just happened?
- **Batch inference is consistently faster than serial** even on CPU because it amortises tokenization and minimises Python loop overhead.
- On a GPU the speedup is typically 5–15× for classification tasks.
- `top_k=None` replaces the deprecated `return_all_scores=True` — it returns confidence scores for all labels, essential for calibration analysis.
- `max_length=128` is deliberately short here to keep the demo fast; in production match the length used during fine-tuning.

In [ ]:
# Challenge: Days 1–12 Audit + OOM Calculator
#
# Part A — Audit your notebook history
# Fill in the paths to your actual fine-tuning output directories below.
# The audit_run_directory function from Step 5 will check each one.
#
# Part B — OOM calculator
# Given: model has H hidden size, L layers, S sequence length, B batch size
# Approximate activation memory (bytes) ≈ B * S * H * L * 4 * 2  (fp32, forward+backward)
# Implement the calculator and find the max safe batch size for a 16 GB GPU.

# ── Part A: fill in your run directories ────────────────────────────────────
my_run_dirs = [
    # "/content/drive/MyDrive/day-03-bert-run",
    # "/content/drive/MyDrive/day-07-ner-run",
    # Add your actual directories here
]

for d in my_run_dirs:
    print(f"\nAuditing: {d}")
    report = audit_run_directory(d)
    for check, passed in report.items():
        print(f"  {'✓' if passed else '✗'}  {check}")

if not my_run_dirs:
    print("Add your fine-tuning output directories to my_run_dirs above.")

# ── Part B: OOM calculator ───────────────────────────────────────────────────
def activation_memory_gb(
    batch_size: int,
    seq_len: int,
    hidden_size: int,
    num_layers: int,
    fp16: bool = True,
) -> float:
    # YOUR IMPLEMENTATION HERE
    # Hint: bytes = B * S * H * L * bytes_per_param * 2 (fwd + bwd)
    # bytes_per_param: fp32 = 4, fp16 = 2
    pass

# Test: bert-base (H=768, L=12), seq=512, fp16
# mem = activation_memory_gb(batch_size=32, seq_len=512, hidden_size=768, num_layers=12, fp16=True)
# print(f"\nEstimated activation memory: {mem:.2f} GB")
# Find the max batch size that fits in 16 GB
# for bs in range(1, 128):
#     if activation_memory_gb(bs, 512, 768, 12, fp16=True) > 16:
#         print(f"Max safe batch size: {bs - 1}")
#         break

---
## Day 13 key concepts recap

| Concept | What to remember |
|---|---|
| `pad_token` for GPT-2 | Set `tokenizer.pad_token = tokenizer.eos_token` before any batched call |
| Label shift in causal LM | Labels = input_ids shifted left by 1; `ignore_index=-100` for padding |
| OOM: batch size | Halve `per_device_train_batch_size`; double `gradient_accumulation_steps` |
| OOM: sequence length | Memory ∝ S²; truncate aggressively during development |
| OOM: fp16 | `fp16=True` in `TrainingArguments` halves activation memory |
| Tokenizer save | `tokenizer.save_pretrained(output_dir)` — always alongside model |
| Reload consistency | Reload tokenizer from `output_dir`, not from original Hub checkpoint |
| Pipeline best practices | Explicit `device=`, `truncation=True`, `batch_size`, `top_k=None` |

> **Tip:** Always call `tokenizer.save_pretrained(output_dir)` alongside `model.save_pretrained(output_dir)` — loading a model without its matching tokenizer is a silent bug that causes subtle mismatches.

---
## What's next
**Day 14 (Capstone)** → Fine-tune a BERT-family model on a task of your choice, evaluate with the correct metric, push the model + tokenizer to the Hub with a complete model card, and deploy a live Gradio Space.

Mark Day 13 complete in your [tracker](../index.html).